In [58]:
import numpy as np
import xarray as xr


from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.models import LinearColorMapper, ColorBar


output_notebook()


# --------------------------------------------------------------------
# 1. Load and subset, then ensure latitude is ascending (south -> north)
# --------------------------------------------------------------------
adt = (
    xr.open_dataset("/home/databot/share/www/data/adt_latest.nc")
      .sel(latitude=slice(-40, -30), longitude=slice(5, 25))
      .squeeze()
)


# Make latitude strictly increasing so that row 0 = southernmost, last row = northernmost
adt = adt.sortby("latitude")   # critical line if source has lat decreasing[web:77]


var = adt["sla"]               # or your SLA variable
lat = adt["latitude"].values
lon = adt["longitude"].values
data = var.values              # 2D (lat, lon), now south->north


# --------------------------------------------------------------------
# 2. Convert lon/lat grid to Web Mercator (EPSG:3857)
# --------------------------------------------------------------------
R = 6378137.0


def lonlat_to_mercator(lon_deg, lat_deg):
    lon_rad = np.radians(lon_deg)
    lat_rad = np.radians(lat_deg)
    x = R * lon_rad
    y = R * np.log(np.tan(np.pi/4 + lat_rad/2))
    return x, y


lon2d, lat2d = np.meshgrid(lon, lat)
x2d, y2d = lonlat_to_mercator(lon2d, lat2d)


x_min, x_max = np.nanmin(x2d), np.nanmax(x2d)
y_min, y_max = np.nanmin(y2d), np.nanmax(y2d)


# --------------------------------------------------------------------
# 3. Figure with mercator axes and tiles
# --------------------------------------------------------------------
p = figure(
    x_axis_type="mercator",
    y_axis_type="mercator",
    x_range=(x_min, x_max),
    y_range=(y_min, y_max),
    width=900,
    height=500,
    title="ADT (latest)"
)


p.add_tile("CartoDB Positron")   # works in recent Bokeh[web:65][web:71]


from bokeh.palettes import Viridis256, linear_palette
from bokeh.models.tickers import FixedTicker

# --------------------------------------------------------------------
# 4. More color blocks (e.g. 0.1 steps from -1 to 1)
# --------------------------------------------------------------------
vmin, vmax = -1.0, 1.0
step = 0.1

edges = np.arange(vmin, vmax + step, step)   # -1.0 ... 1.0  (21 edges -> 20 bins)
centers = (edges[:-1] + edges[1:]) / 2       # 20 centers

# Quantize data into 0.1 steps so each band is uniform
data_q = np.clip(
    np.round(data / step) * step,
    vmin + step/2,
    vmax - step/2
)

# Build a 20-color palette from a 256-color base[web:96][web:107]
palette = linear_palette(Viridis256, len(centers))

color_mapper = LinearColorMapper(
    palette=palette,
    low=vmin,
    high=vmax,
    nan_color=(0, 0, 0, 0),
)

# --------------------------------------------------------------------
# 5. Use quantized data and fixed ticks
# --------------------------------------------------------------------
p.image(
    image=[data_q],
    x=x_min,
    y=y_min,
    dw=x_max - x_min,
    dh=y_max - y_min,
    color_mapper=color_mapper,
)

ticks = np.round(edges, 1)

color_bar = ColorBar(
    color_mapper=color_mapper,
    ticker=FixedTicker(ticks=list(ticks)),
    width=8,
)
p.add_layout(color_bar, "right")


show(p)


Loading BokehJS ...

In [24]:
# ----------------- CONFIG -----------------

ADT_PATH = "/home/databot/share/www/data/adt_latest.nc"
SG675_PATH = "/home/databot/share/www/data/SG675_GOUGH_SAMBA/sg675_SG675_Gough_SAMBA_timeseries.nc"
WG1070_PATH = "/home/databot/share/www/data/waveglider/GOUGH_SAMBA_2025/waveglider-telemetry.nc"
SWOT_GEOJSON = "../../data/swot/KaRIn_2kms_science_geometries.geojson"
SWOT_CSV = "../../data/swot/selected_passes.csv"

bextent = [10, 20, -38, -32]  # [lon_min, lon_max, lat_min, lat_max]

R = 6378137.0

def lonlat_to_mercator(lon, lat):
    lon = np.asarray(lon, dtype=float)
    lat = np.asarray(lat, dtype=float)
    x = np.deg2rad(lon) * R
    y = np.log(np.tan(np.pi/4 + np.deg2rad(lat)/2)) * R
    return x, y

In [25]:
# ----------------- LOAD AVISO + DERIVE FIELDS -----------------

adt = xr.open_dataset(ADT_PATH).sel(
    latitude=slice(-60, -20),
    longitude=slice(0, 35)
).squeeze()

adt = adt.assign(
    gos=np.sqrt(adt.ugos**2 + adt.vgos**2)
)

lon2d, lat2d = np.meshgrid(adt.longitude.values, adt.latitude.values)
lon_flat = lon2d.ravel()
lat_flat = lat2d.ravel()

sla_flat = adt.sla.values.ravel()
gos_flat = adt.gos.values.ravel()
ugos_flat = adt.ugos.values.ravel()
vgos_flat = adt.vgos.values.ravel()

x_merc, y_merc = lonlat_to_mercator(lon_flat, lat_flat)

# background sources
src_sla = ColumnDataSource(dict(
    x=x_merc,
    y=y_merc,
    val=sla_flat,
    lon=lon_flat,
    lat=lat_flat,
    ugos=ugos_flat,
    vgos=vgos_flat,
))

src_gos = ColumnDataSource(dict(
    x=x_merc,
    y=y_merc,
    val=gos_flat,
    lon=lon_flat,
    lat=lat_flat,
    ugos=ugos_flat,
    vgos=vgos_flat,
))

# color mappers
mapper_sla = LinearColorMapper(
    palette="RdBu11",
    low=-1.0,
    high=1.0,
)
mapper_gos = LinearColorMapper(
    palette="Viridis256",
    low=0.0,
    high=1.5,
)

# ----------------- SG675 TRACK -----------------

ds_sg675 = xr.open_dataset(SG675_PATH)

sg675_lon = ds_sg675.end_longitude.values.astype(float)
sg675_lat = ds_sg675.end_latitude.values.astype(float)
sg675_x, sg675_y = lonlat_to_mercator(sg675_lon, sg675_lat)

src_sg675 = ColumnDataSource(dict(
    x=sg675_x,
    y=sg675_y,
    lon=sg675_lon,
    lat=sg675_lat,
))

# ----------------- WG1070 TRACK -----------------

ds_wg = xr.open_dataset(WG1070_PATH)

wg_lon = ds_wg.longitude.values.astype(float)
wg_lat = ds_wg.latitude.values.astype(float)
wg_x, wg_y = lonlat_to_mercator(wg_lon, wg_lat)

src_wg = ColumnDataSource(dict(
    x=wg_x,
    y=wg_y,
    lon=wg_lon,
    lat=wg_lat,
))

# just your “last” WG position
wg_last_lon = 15.41
wg_last_lat = -34.35
wg_last_x, wg_last_y = lonlat_to_mercator([wg_last_lon], [wg_last_lat])
src_wg_last = ColumnDataSource(dict(
    x=wg_last_x,
    y=wg_last_y,
    lon=[wg_last_lon],
    lat=[wg_last_lat],
))

# ----------------- SG267 (GPS fixes) -----------------

# Keep your glob/xr loop but flatten to Bokeh source
from glob import glob
sg267_files = sorted(glob("/home/databot/share/www/data/sg267_WHIRLS_Mission2_2026/*.nc"))

sg267_lat_fix = []
sg267_lon_fix = []

for fname in sg267_files:
    ds = xr.open_dataset(fname, decode_timedelta=False)
    sg267_lat_fix.append(ds["log_gps_lat"][2].item())
    sg267_lon_fix.append(ds["log_gps_lon"][2].item())

sg267_lon_fix = np.array(sg267_lon_fix, dtype=float)
sg267_lat_fix = np.array(sg267_lat_fix, dtype=float)
sg267_x, sg267_y = lonlat_to_mercator(sg267_lon_fix, sg267_lat_fix)

src_sg267 = ColumnDataSource(dict(
    x=sg267_x,
    y=sg267_y,
    lon=sg267_lon_fix,
    lat=sg267_lat_fix,
))

src_sg267_last = ColumnDataSource(dict(
    x=[sg267_x[-1]],
    y=[sg267_y[-1]],
    lon=[sg267_lon_fix[-1]],
    lat=[sg267_lat_fix[-1]],
))

# ----------------- SWOT SWATHS (recent + upcoming) -----------------

now = datetime.now(timezone.utc)

swaths = gpd.read_file(SWOT_GEOJSON)
if swaths.crs is None:
    swaths = swaths.set_crs(epsg=4326)
else:
    swaths = swaths.to_crs(epsg=4326)

bbox_poly = box(bextent[0], bextent[2], bextent[1], bextent[3])

try:
    candidate_idx = list(swaths.sindex.query(bbox_poly, predicate="intersects"))
    swaths_in_box = swaths.iloc[candidate_idx]
    swaths_in_box = swaths_in_box[swaths_in_box.intersects(bbox_poly)]
except Exception:
    swaths_in_box = swaths[swaths.intersects(bbox_poly)]

df_passes = pd.read_csv(SWOT_CSV, sep=";")
df_passes["First date"] = pd.to_datetime(df_passes["First date"], utc=True)
df_passes["Last date"] = pd.to_datetime(df_passes["Last date"], utc=True)

past = df_passes[df_passes["Last date"] <= now]
recent = past.iloc[-1] if not past.empty else None

future = df_passes[df_passes["First date"] > now]
upcoming = future.iloc[0] if not future.empty else None

def swath_to_src(pass_num):
    gsub = swaths_in_box.loc[swaths_in_box["pass_number"] == pass_num]
    if gsub.empty:
        return ColumnDataSource(dict(xs=[], ys=[]))

    xs_list = []
    ys_list = []
    for geom in gsub.geometry.values:
        longitudes, latitudes = geom.exterior.coords.xy
        x, y = lonlat_to_mercator(longitudes, latitudes)
        xs_list.append(list(x))
        ys_list.append(list(y))
    return ColumnDataSource(dict(xs=xs_list, ys=ys_list))

src_swot_recent = swath_to_src(int(recent["Pass number"])) if recent is not None else ColumnDataSource(dict(xs=[], ys=[]))
src_swot_upcoming = swath_to_src(int(upcoming["Pass number"])) if upcoming is not None else ColumnDataSource(dict(xs=[], ys=[]))

In [30]:
import numpy as np
from pyproj import Transformer
from bokeh.plotting import figure, show
from bokeh.tile_providers import get_provider, CARTODBPOSITRON
from bokeh.models import LinearColorMapper, ColorBar
from bokeh.palettes import RdBu11  # diverging for anomalies

# ---- 1) Convert lon/lat grid to Web Mercator bounds ----
transformer = Transformer.from_crs("epsg:4326", "epsg:3857", always_xy=True)

lon_min, lon_max = lon.min(), lon.max()
lat_min, lat_max = lat.min(), lat.max()

x_min, y_min = transformer.transform(lon_min, lat_min)
x_max, y_max = transformer.transform(lon_max, lat_max)

# Bokeh's image glyph expects a list of 2D arrays; flip vertically if needed
img = [np.flipud(sla)]

# ---- 2) Set up map figure ----
tile_provider = get_provider(CARTODBPOSITRON)  # base map tiles[web:1]

p = figure(
    x_range=(x_min, x_max),
    y_range=(y_min, y_max),
    x_axis_type="mercator",
    y_axis_type="mercator",
    width=800,
    height=400,
    title="Sea level anomaly",
)

p.add_tile(tile_provider)

# ---- 3) Color mapping for SLA ----
vmin, vmax = np.nanpercentile(sla, [2, 98])  # robust limits
mapper = LinearColorMapper(palette=list(reversed(RdBu11)), low=vmin, high=vmax)

# ---- 4) Plot the SLA image in Web Mercator coordinates ----
# dw, dh are WIDTH and HEIGHT in data units (same CRS as x/y)[web:22][web:26]
p.image(
    image=img,
    x=x_min,
    y=y_min,
    dw=(x_max - x_min),
    dh=(y_max - y_min),
    color_mapper=mapper,
)

color_bar = ColorBar(color_mapper=mapper, title="SLA (m)")
p.add_layout(color_bar, "right")

show(p)


ModuleNotFoundError: No module named 'bokeh.tile_providers'

In [29]:
# ----------------- FIGURE -----------------

p = figure(
    x_axis_type="mercator",
    y_axis_type="mercator",
    width=950,
    height=650,
    match_aspect=True,
    tools="pan,wheel_zoom,box_zoom,reset,save",
    title=f"AVISO + Gliders {np.datetime_as_string(adt.time.values, unit='D')}",
)

# Bokeh ≥3: pass name directly
p.add_tile("CARTODBPOSITRON")

# set extent
x0, y0 = lonlat_to_mercator(bextent[0], bextent[2])
x1, y1 = lonlat_to_mercator(bextent[1], bextent[3])
p.x_range.start, p.x_range.end = x0, x1
p.y_range.start, p.y_range.end = y0, y1

# SLA background
r_sla = p.square(
    "x", "y",
    source=src_sla,
    size=6,
    line_color=None,
    fill_color={"field": "val", "transform": mapper_sla},
    fill_alpha=0.9,
)
cb_sla = ColorBar(
    color_mapper=mapper_sla,
    title="SLA (m)",
    label_standoff=8,
)
p.add_layout(cb_sla, "right")

In [ ]:










# |u_g| background
r_gos = p.square(
    "x", "y",
    source=src_gos,
    size=6,
    line_color=None,
    fill_color={"field": "val", "transform": mapper_gos},
    fill_alpha=0.9,
)
cb_gos = ColorBar(
    color_mapper=mapper_gos,
    title="|u_g| (m s⁻¹)",
    label_standoff=8,
)
p.add_layout(cb_gos, "right")

# start with SLA visible
r_gos.visible = False
cb_gos.visible = False

# crude vector field (segments)
scale = 1.2e5
u = ugos_flat
v = vgos_flat
dx = (u / np.maximum(gos_flat, 1e-6)) * scale * gos_flat
dy = (v / np.maximum(gos_flat, 1e-6)) * scale * gos_flat

src_vec = ColumnDataSource(dict(
    x0=x_merc,
    y0=y_merc,
    x1=x_merc + dx,
    y1=y_merc + dy,
))
r_vec = p.segment(
    "x0", "y0", "x1", "y1",
    source=src_vec,
    line_color="black",
    line_alpha=0.5,
    line_width=1,
)

# ----------------- GLIDER / WG RENDERERS -----------------

r_sg675 = p.line(
    "x", "y", source=src_sg675,
    line_color="0.5",
    line_width=2,
)
r_wg_track = p.circle(
    "x", "y", source=src_wg,
    size=4,
    fill_color="C1",
    line_color=None,
)
r_wg_last = p.circle(
    "x", "y", source=src_wg_last,
    size=8,
    fill_color="C1",
    line_color="black",
)

r_sg267 = p.circle(
    "x", "y", source=src_sg267,
    size=5,
    fill_color="gold",
    line_color="gold",
)
r_sg267_last = p.circle(
    "x", "y", source=src_sg267_last,
    size=9,
    fill_color="gold",
    line_color="black",
)

p.add_tools(HoverTool(
    renderers=[r_sg675],
    tooltips=[("SG675", "@lon{0.000}, @lat{0.000}")]
))
p.add_tools(HoverTool(
    renderers=[r_wg_track],
    tooltips=[("Wave Glider", "@lon{0.000}, @lat{0.000}")]
))
p.add_tools(HoverTool(
    renderers=[r_sg267],
    tooltips=[("SG267", "@lon{0.000}, @lat{0.000}")]
))

# ----------------- SWOT SWATH PATCHES -----------------

r_swot_recent = p.patches(
    "xs", "ys",
    source=src_swot_recent,
    fill_color="black",
    fill_alpha=0.45,
    line_color="black",
    line_width=0.8,
)
r_swot_upcoming = p.patches(
    "xs", "ys",
    source=src_swot_upcoming,
    fill_color="black",
    fill_alpha=0.20,
    line_color="black",
    line_width=0.8,
)

# ----------------- LEGEND -----------------

legend_items = []

if recent is not None:
    legend_items.append(LegendItem(
        label=f"Last SWOT Pass {int(recent['Pass number'])} — {recent['First date'].strftime('%Y-%m-%d %H:%M')}",
        renderers=[r_swot_recent],
    ))
if upcoming is not None:
    legend_items.append(LegendItem(
        label=f"Next SWOT Pass {int(upcoming['Pass number'])} — {upcoming['First date'].strftime('%Y-%m-%d %H:%M')}",
        renderers=[r_swot_upcoming],
    ))

legend_items.extend([
    LegendItem(label="SG675 Mission #1", renderers=[r_sg675]),
    LegendItem(label="Wave Glider Mission #1", renderers=[r_wg_track]),
    LegendItem(label="Wave Glider Mission #2 Last Pos", renderers=[r_wg_last]),
    LegendItem(label="SG267 Mission #2", renderers=[r_sg267]),
    LegendItem(label="SG267 Mission #2 Last Pos", renderers=[r_sg267_last]),
])

legend = Legend(items=legend_items, location="bottom_left")
p.add_layout(legend, "below")
p.legend.click_policy = "hide"

# ----------------- SLA ↔ GOS TOGGLE -----------------

toggle = RadioButtonGroup(labels=["SLA", "GOS"], active=0)

def _toggle_cb(attr, old, new):
    if toggle.active == 0:
        r_sla.visible = True
        cb_sla.visible = True
        r_gos.visible = False
        cb_gos.visible = False
    else:
        r_sla.visible = False
        cb_sla.visible = False
        r_gos.visible = True
        cb_gos.visible = True

toggle.on_change("active", _toggle_cb)

curdoc().add_root(column(toggle, p))
curdoc().title = "AVISO + Gliders Mission Map"


ValueError: failed to validate Line(id='p1109', ...).line_color: expected either None or a value of type Color, got '0.5'